In [3]:
# ==============================================================================
# Step 1: Import All Necessary Libraries
# ==============================================================================
import pandas as pd
import numpy as np
import warnings
import joblib

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
import xgboost as xgb
import lightgbm as lgb

# Evaluation
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Configure settings
warnings.filterwarnings('ignore')
print("--- Step 1: Libraries imported successfully ---")


# ==============================================================================
# Step 2: Load and Prepare the Dataset
# ==============================================================================
print("\n--- Step 2: Loading and preparing the dataset ---")

file_path = 'processed_data.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Dataset '{file_path}' loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"\nERROR: Could not find '{file_path}'.")
    print("Please make sure the dataset is in the same folder as the notebook.")
    # In a script, you might exit, but in a notebook, we can just stop.
    # We will raise the error to stop the cell execution.
    raise

if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

# Critical cleaning of the 'YIELD' column
print("\nCleaning the target variable 'YIELD'...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
initial_rows = len(df)
df.dropna(subset=['YIELD'], inplace=True)
final_rows = len(df)
print(f"Dropped {initial_rows - final_rows} rows with missing YIELD values.")

# Handle missing values in feature columns
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)
    
print("Missing values handled.")


# ==============================================================================
# Step 3: Feature Engineering (One-Hot Encoding)
# ==============================================================================
print("\n--- Step 3: Performing one-hot encoding ---")

categorical_cols = df.select_dtypes(include=['object']).columns
if 'YIELD' in categorical_cols:
    categorical_cols = categorical_cols.drop('YIELD')

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f"Shape after one-hot encoding: {df_encoded.shape}")


# ==============================================================================
# Step 4: Data Splitting and Scaling
# ==============================================================================
print("\n--- Step 4: Preparing final data for modeling ---")

X = df_encoded.drop('YIELD', axis=1)
y = df_encoded['YIELD']

X_train_pre, X_test_pre, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_pre)
X_test = scaler.transform(X_test_pre)

print(f"Data split and scaled. Training data shape: {X_train.shape}")


# ==============================================================================
# Step 5: Train High-Performance Models
# ==============================================================================
print("\n--- Step 5: Training high-performance models ---")

# --- Model 1: XGBoost ---
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, learning_rate=0.05,
                           n_jobs=-1, random_state=42, early_stopping_rounds=50)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost training complete.")

# --- Model 2: LightGBM ---
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(objective='regression', n_estimators=1000, learning_rate=0.05,
                            n_jobs=-1, random_state=42)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='rmse',
            callbacks=[lgb.early_stopping(50, verbose=False)])
y_pred_lgb = lgb_model.predict(X_test)
print("LightGBM training complete.")


# ==============================================================================
# Step 6: Evaluate Model Performance
# ==============================================================================
print("\n--- Step 6: Evaluating model performance ---")

full_results = {}
full_results['LightGBM'] = [
    r2_score(y_test, y_pred_lgb),
    np.sqrt(mean_squared_error(y_test, y_pred_lgb)),
    mean_absolute_error(y_test, y_pred_lgb)
]
full_results['XGBoost'] = [
    r2_score(y_test, y_pred_xgb),
    np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
    mean_absolute_error(y_test, y_pred_xgb)
]

full_results_df = pd.DataFrame.from_dict(full_results,
                                         orient='index',
                                         columns=['R-squared (R²)', 'RMSE', 'MAE'])
full_results_df = full_results_df.sort_values(by='R-squared (R²)', ascending=False)

print("\n--- FINAL MODEL PERFORMANCE COMPARISON ---")
print(full_results_df)


# ==============================================================================
# Step 7: Save the Best Model
# ==============================================================================
print("\n--- Step 7: Saving the best model ---")

best_model_name = full_results_df.index[0]
best_model_filename = "yield_prediction_model.joblib"

if best_model_name == 'LightGBM':
    joblib.dump(lgb_model, best_model_filename)
    print(f"Best model ('LightGBM') saved as '{best_model_filename}'")
else:
    joblib.dump(xgb_model, best_model_filename)
    print(f"Best model ('XGBoost') saved as '{best_model_filename}'")

print("\n--- Process Complete ---")

--- Step 1: Libraries imported successfully ---

--- Step 2: Loading and preparing the dataset ---
Dataset 'processed_data.csv' loaded successfully. Shape: (114693, 61)

Cleaning the target variable 'YIELD'...
Dropped 36353 rows with missing YIELD values.
Missing values handled.

--- Step 3: Performing one-hot encoding ---
Shape after one-hot encoding: (78340, 338)

--- Step 4: Preparing final data for modeling ---
Data split and scaled. Training data shape: (62672, 337)

--- Step 5: Training high-performance models ---
Training XGBoost...
XGBoost training complete.
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014568 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6627
[LightGBM] [Info] Number of data points in the train set: 62672, number of used features: 333
[LightGBM] [Info] Start training from score 1092.690